In [1]:
import jax.numpy as jnp

In [3]:
def relu(x):
    return jnp.maximum(0,x)
def softmax(x):
    return jnp.exp(x)/jnp.sum(jnp.exp(x),axis=0)

# L=−[y⋅log(y^)+(1−y)⋅log(1−y^)]

def cross_entropy(x,y):
    return -jnp.sum(y*jnp.log(x))

In [4]:
x=jnp.arange(6.0)
print(relu(x))
print(softmax(x))
print(cross_entropy(softmax(x),x))

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


[0. 1. 2. 3. 4. 5.]
[0.00426978 0.01160646 0.03154963 0.08576079 0.233122   0.6336913 ]
26.8429


In [5]:
from jax import random

key=random.PRNGKey(0)
print(key)
key,subkey = random.split(key)
print(key)
print(subkey)

[0 0]
[4146024105  967050713]
[2718843009 1272950319]


In [6]:
# 功能：生成符合标准正态分布（均值为 0，标准差为 1）的随机数数组(包含 100 万个元素的一维数组)
x = random.normal(key,(1_000_000,))
# IPython/Jupyter 环境中的魔术命令.多次运行后面的代码，并计算平均执行时间
# %timeit relu(x)

In [7]:
from jax import jit

jitted_relu = jit(relu)
_ = jitted_relu(x)  #complies on first call
%timeit relu(x)         #未即时编译
%timeit jitted_relu(x)  #即时编译

223 μs ± 7.59 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
173 μs ± 9.01 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [8]:
# 自定义一个激活函数进行性能测试
def selu(x,alpha=1.67,lmbda=1.05):
    return lmbda * jnp.where(x>0,x,alpha*jnp.exp(x)-alpha)

key = random.key(1000)
x = random.normal(key,(1_000_000,))
%timeit selu(x).block_until_ready()      #未即时编译

selu_jit = jit(selu)
_=selu_jit(x) #complies on first call
%timeit selu_jit(x).block_until_ready()  #即时编译


1.51 ms ± 57.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
259 μs ± 19.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [10]:
# matrix multiplication(矩阵乘法测试)
def mm(x,y):
    return jnp.dot(x,y)

@jit
def mm_jit(x,y):
    return jnp.dot(x,y)


In [12]:
a = random.normal(key,(1000,1000))
b = random.normal(key,(1000,1000))

%timeit mm(a,b)     #未即时编译
%timeit mm_jit(a,b) #即时编译

3.88 ms ± 189 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
4.02 ms ± 376 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
